In [2]:
import os
import re
import numpy as np
from google import genai
from dotenv import load_dotenv

load_dotenv(override=True)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "YOUR_API_KEY_HERE")
client = genai.Client(api_key=GEMINI_API_KEY)

GEN_MODEL = "gemini-2.5-flash"
EMBED_MODEL = "models/gemini-embedding-001"  #  embedding model name

print("Client ready.")

Client ready.


In [3]:
# Step 1: List available text files in the current directory
import glob

current_dir = os.getcwd()
print(f"Current directory: {current_dir}\n")

# Find all .txt files
txt_files = glob.glob("pinnacle_trust_bank_policy.txt")

if txt_files:
    print(f"📄 Found {len(txt_files)} text file(s):")
    for i, file in enumerate(txt_files, 1):
        file_size = os.path.getsize(file)
        print(f"  {i}. {file} ({file_size:,} bytes)")
    
    print(f"\n📌 Will use: {txt_files[0]} for RAG demonstration")
else:
    print("❌ No .txt files found in current directory")
    print("💡 Please add a .txt file to the current directory and run this cell again.")

Current directory: e:\Gen AI\Gen AI training_Anjishnu\Gen AI

📄 Found 1 text file(s):
  1. pinnacle_trust_bank_policy.txt (80,645 bytes)

📌 Will use: pinnacle_trust_bank_policy.txt for RAG demonstration


In [6]:

#Step 2: Load and chunk the selected text file
selected_file = txt_files[0]
print(f"📄 Loading file: {selected_file}\n")

# Read the file content
with open(selected_file, "r", encoding="utf-8") as f:
    file_content = f.read()

print(f"File size: {len(file_content)} characters")
print(f"Preview (first 200 chars):\n{file_content[:200]}...\n")

# Chunk the file content with larger chunks for better context
def chunk_text_file(text, chunk_size=1000, overlap=100):
    """Split text into overlapping chunks"""
    text = text.strip()
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)

        if end >= len(text):
            break

        start = end - overlap

    return chunks


# Create chunks from file
file_chunks = chunk_text_file(file_content, chunk_size=1000, overlap=100)

# Create chunk records with metadata
file_chunk_records = []
for idx, chunk_text in enumerate(file_chunks, 1):
    file_chunk_records.append({
        "chunk_id": f"F{idx}",
        "source": selected_file,
        "text": chunk_text.strip()
    })

print(f"🧩 Created {len(file_chunk_records)} chunks from the file\n")

print("Sample chunks:")
for i, rec in enumerate(file_chunk_records[:3], 1):
    print(f"\n[{rec['chunk_id']}] ({rec['source']})")
    print(f"  Text: {rec['text'][:100]}...")

📄 Loading file: pinnacle_trust_bank_policy.txt

File size: 80551 characters
Preview (first 200 chars):
PINNACLE TRUST BANK PLC
COMPREHENSIVE BANKING POLICIES & CUSTOMER SERVICE HANDBOOK
Document Version 4.2  |  Effective D...

🧩 Created 90 chunks from the file

Sample chunks:

[F1] (pinnacle_trust_bank_policy.txt)
  Text: ================================================================================
PINNACLE TRUST BANK...

[F2] (pinnacle_trust_bank_policy.txt)
  Text: , and 1,200+ ATMs spread across all major cities and towns.

PTB is regulated by the Metropolis Fina...

[F3] (pinnacle_trust_bank_policy.txt)
  Text: believe in providing equal access to financial services regardless of background.

Responsibility: W...


In [7]:
# Step 3: Create embeddings for all file chunks
print("🔄 Creating embeddings for file chunks...")

def embed_texts(texts, model=EMBED_MODEL):
    """Generate embeddings for a list of texts using the Gemini embedding model."""
    result = client.models.embed_content(contents=texts, model=model)

    #Extract embeddings from the response
    embeddings = []
    if hasattr(result, 'embeddings'):
        for emb in result.embeddings:
            if hasattr(emb, 'values'):
                embeddings.append(np.array(emb.values, dtype=np.float32))
    return embeddings

file_texts = [rec["text"] for rec in file_chunk_records]
file_vectors = embed_texts(file_texts)

print(f"✅ Created {len(file_vectors)} embeddings")
print(f"📏 Embedding dimension: {len(file_vectors[0])}")
print("🚀 Ready for semantic search!") 

🔄 Creating embeddings for file chunks...
✅ Created 90 embeddings
📏 Embedding dimension: 3072
🚀 Ready for semantic search!


In [8]:
# Step 4: Create a retrieval function for the text file
def retrieve_from_file(query, top_k=3):
    """Retrieve most relevant chunks from the file based on query"""
    
    # Get query embedding
    query_vector = embed_texts([query])[0]
    
    # Calculate similarity scores
    scored_chunks = []
    for rec, vec in zip(file_chunk_records, file_vectors):
        score = cosine_similarity(query_vector, vec)
        scored_chunks.append((score, rec))
    
    # Sort by score and return top_k
    scored_chunks.sort(key=lambda x: x[0], reverse=True)
    return scored_chunks[:top_k]


# Test the retrieval with a sample query
test_query = "How to apply for a loan?"
print(f"🔍 Query: {test_query}\n")

def cosine_similarity(a, b):
    """Calculate cosine similarity between two vectors (0=unrelated, 1=identical)."""
    denom=(np.linalg.norm(a) * np.linalg.norm(b))
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)


results = retrieve_from_file(test_query, top_k=3)

print("📊 Top 3 relevant chunks:\n")
for rank, (score, rec) in enumerate(results, 1):
    print(f"{rank}. Score: {score:.4f} | [{rec['chunk_id']}]")
    print(f"   Text: {rec['text'][:150]}...")
    print()

🔍 Query: How to apply for a loan?

📊 Top 3 relevant chunks:

1. Score: 0.6904 | [F23]
   Text: Products menu on the home screen.
Step 4: Select your preferred loan type and complete the guided application form within the app.
Step 5: Upload docu...

2. Score: 0.6714 | [F22]
   Text: form with accurate personal, employment, and financial information.
Step 4: Upload the required supporting documents in PDF or JPG format. Each file m...

3. Score: 0.6712 | [F24]
   Text: ft for the processing fee, drawn in favor of "Pinnacle Trust Bank PLC."
Step 6: Obtain the acknowledgement receipt from the branch officer. This recei...



In [9]:
# Step 5: Implement RAG - Retrieval-Augmented Generation
def rag_answer_from_file(query, top_k=3):
    """
    RAG pipeline:
    1. Retrieve relevant chunks from file
    2. Build context from retrieved chunks
    3. Generate answer using LLM with context
    """

    # Retrieve relevant chunks
    results = retrieve_from_file(query, top_k=top_k)

    # Build context block
    context_parts = []
    for score, rec in results:
        context_parts.append(
            f"[{rec['chunk_id']}] (relevance: {score:.2f})\n{rec['text']}"
        )

    context_block = "\n\n".join(context_parts)

    # Create prompt for LLM
    prompt = f"""You are a helpful assistant that answers questions based ONLY on the provided context.

Instructions:
- Answer the question using ONLY information from the context below
- If the context doesn’t contain enough information, say "I cannot answer based on the provided context"
- Include citations like [F1], [F2] to reference specific chunks
- Be concise and accurate

Question: {query}

Context from file:
{context_block}

Answer:"""

    # Generate answer using Gemini
    response = client.models.generate_content(
        model=GEN_MODEL,
        contents=prompt
    )

    answer = response.text

    # Extract citations used
    import re
    citations = sorted(set(re.findall(r"\[F\d+\]", answer)))

    return {
        "answer": answer,
        "query": query,
        "retrieved_chunks": results,
        "citations": citations
    }


print("✅ RAG function ready!")

✅ RAG function ready!


In [10]:
# Step 6: Test RAG with sample questions
print("=" * 80)
print("🤖 RAG DEMONSTRATION - Question Answering from Text File")
print("=" * 80)

# List of test questions
test_questions = [
    "What are the key features of Python?",
    "How is Python used in machine learning?",
    "What are some best practices for Python programming?"
]

for i, question in enumerate(test_questions, 1):
    print(f"\n{'=' * 80}")
    print(f"Question {i}: {question}")
    print("=" * 80)

    # Get RAG answer
    result = rag_answer_from_file(question, top_k=3)

    print(f"\n📄 Answer:")
    print(result["answer"])

    print(f"\n🔗 Citations used: {', '.join(result['citations']) if result['citations'] else 'None'}")

    print(f"\n📊 Retrieved evidence:")
    for rank, (score, rec) in enumerate(result["retrieved_chunks"], 1):
        print(f"  {rank}. [{rec['chunk_id']}] Relevance: {score:.4f}")
        print(f"     Preview: {rec['text'][:100]}...")

print(f"\n{'=' * 80}")
print("✅ RAG demonstration complete!")
print("=" * 80)

🤖 RAG DEMONSTRATION - Question Answering from Text File

Question 1: What are the key features of Python?

📄 Answer:
I cannot answer based on the provided context.

🔗 Citations used: None

📊 Retrieved evidence:
  1. [F38] Relevance: 0.5489
     Preview: ========================================================================

PTB's digital banking ecos...
  2. [F18] Relevance: 0.5003
     Preview: arty guarantee from a creditworthy individual is required. For loans above $40,000, tangible collate...
  3. [F65] Relevance: 0.4999
     Preview: ce on the PTB website, at least 30 calendar days before the changes take effect.

For changes mandat...

Question 2: How is Python used in machine learning?

📄 Answer:
I cannot answer based on the provided context.

🔗 Citations used: None

📊 Retrieved evidence:
  1. [F66] Relevance: 0.5050
     Preview: -augmented generation (RAG) system training and evaluation.


SECTION A: LOAN ELIGIBILITY AND APPLIC...
  2. [F25] Relevance: 0.5041
     Preview: 

In [11]:
# Step 6: Test RAG with sample questions
print("=" * 80)
print("🤖 RAG DEMONSTRATION - Question Answering from Text File")
print("=" * 80)

test_questions = [
    "How to apply for a loan?",
    "Any customer care details?"
]

for i, question in enumerate(test_questions, 1):
    print(f"\n{'=' * 80}")
    print(f"Question {i}: {question}")
    print("=" * 80)

    # Get RAG answer
    result = rag_answer_from_file(question, top_k=3)

    print(f"\n📄 Answer:")
    print(result["answer"])

    print(f"\n🔗 Citations used: {', '.join(result['citations']) if result['citations'] else 'None'}")

    print(f"\n📊 Retrieved evidence:")
    for rank, (score, rec) in enumerate(result["retrieved_chunks"], 1):
        print(f"  {rank}. [{rec['chunk_id']}] Relevance: {score:.4f}")
        print(f"     Preview: {rec['text'][:100]}...")

print(f"\n{'=' * 80}")
print("✅ RAG demonstration complete!")
print("=" * 80)

🤖 RAG DEMONSTRATION - Question Answering from Text File

Question 1: How to apply for a loan?

📄 Answer:
You can apply for a loan through several methods:

**1. Mobile App Application**
*   Download the PTB Mobile App from Google Play Store or Apple App Store [F22].
*   Log in with existing PTB credentials or register using your mobile number and set up a secure PIN [F22].
*   Tap "Apply for a Loan" under the Services or Products menu on the home screen [F22], [F23].
*   Select your preferred loan type and complete the guided application form within the app [F22], [F23].
*   Upload documents using your phone's camera or from your device gallery [F23]. The app supports direct camera capture for clear document images [F23]. Documents should be in PDF or JPG format, not exceeding 5 MB each [F22].
*   Review all entered details, accept the declaration, and submit the application [F22].
*   Track its real-time status under "My Applications" in the app [F23]. You will receive an Application 